In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / 'src').exists():
    raise RuntimeError('Open this notebook from the project root folder.')

from src.transformation.auction_etl import build_auction_table
from src.transformation.build_model_table import build_model_table

In [ ]:
auction = build_auction_table(ROOT)
auction.head()

In [ ]:
auction_quality = auction.groupby('auction_year', as_index=False).agg(
    sold_players=('player_name_auction', 'size'),
    matched_players=('player_id', lambda values: values.notna().sum()),
    median_price_inr=('price_inr', 'median'),
)
auction_quality['match_rate_percent'] = (100 * auction_quality['matched_players'] / auction_quality['sold_players']).round(1)
auction_quality

In [ ]:
model_table = build_model_table(ROOT)
print('Model rows:', len(model_table))
print('Model columns:', len(model_table.columns))
model_table[['auction_year', 'player_name_canonical', 'role', 'price_inr', 'ipl_runs', 't20_runs', 'odi_runs']].head()

In [ ]:
median_prices = auction.groupby('auction_year')['price_inr'].median().div(10_000_000)
median_prices.plot(marker='o', figsize=(9, 4), title='Median sold-player price by auction year')
plt.xlabel('Auction year')
plt.ylabel('Median price (₹ crore)')
plt.grid(alpha=0.25)
plt.show()